**NOTE:** 
- cell type are defined as vulnerable if relative abudance along the CPS.  Does this information exist for each cell type/supertypes examined in the paper?
- 


### Notes on how to read parquet files

Your output files are at `.../density_based_clustering_fulldataset_supertype/{donor}/{specimen}.parquet`. Each row is one transcript, with these columns:

| Column | Meaning |
|---|---|
| `global_x`, `global_y`, `global_z` | position in µm (`global_z` is **already multiplied by 1.5**) |
| `fov`, `gene`, `cell_id` | copied from the original transcript CSV |
| `cell` | the sampled cell this row belongs to |
| `supertype` | that cell's Supertype |
| `set` | `somatic_ds` (soma cluster from the enriched genes) or `all_ds` (all transcripts within 2 µm of the soma) |

Each sampled cell has rows in both sets.

## 1. Read a whole file
```python
import pandas as pd
ROOT = '/mnt/xuisebnas2/Lab/Bereket_SEA_AD_analysis/analysis_result/density_based_clustering_fulldataset_supertype'

df = pd.read_parquet(f'{ROOT}/H21.33.001/1192523794.parquet')
```
The column types come back exactly as saved: float32 coordinates, and categorical `gene`, `supertype` and `set`. You don't need any `dtype=` arguments.

## 2. Read only some columns
```python
df = pd.read_parquet(path, columns=['global_x', 'global_y', 'global_z', 'gene', 'cell', 'set'])
```
The other columns are never read from disk, so this is faster and uses less memory.

## 3. Read only some rows (filters)
```python
# somatic transcripts of one supertype
df = pd.read_parquet(path, filters=[('supertype', '==', 'Sst_2'), ('set', '==', 'somatic_ds')])

# one cell
df = pd.read_parquet(path, filters=[('cell', '==', 1028020725201222103)])

# several values
df = pd.read_parquet(path, filters=[('supertype', 'in', ['Pvalb_1', 'Pvalb_2'])])
```
Conditions in one list are combined with AND. For OR, use a list of lists: `[[cond_a], [cond_b]]`.

## 4. Look inside a file without loading it
```python
import pyarrow.parquet as pq
f = pq.ParquetFile(path)
print(f.schema_arrow)          # column names and types
print(f.metadata.num_rows)     # row count
```

## 5. Read many sections
The donor and specimen are in the file path, not in the columns, so add them as you read:
```python
import glob, os
parts = []
for fp in glob.glob(f'{ROOT}/*/*.parquet'):
    d = pd.read_parquet(fp, filters=[('supertype', '==', 'Sst_2'), ('set', '==', 'somatic_ds')])
    d['donor'] = fp.split(os.sep)[-2]
    d['specimen'] = os.path.basename(fp).removesuffix('.parquet')
    parts.append(d)
sst2 = pd.concat(parts, ignore_index=True)
```
Filter while reading, as above. Loading all ~69 full sections at once would be roughly 200M rows.

## 6. Work cell by cell
```python
for cell_id, c in df[df['set'] == 'somatic_ds'].groupby('cell', observed=True):
    xyz = c[['global_x', 'global_y', 'global_z']].to_numpy()
    genes = c['gene'].to_numpy()
    ...
```
Add `observed=True` when grouping by a categorical column (here `supertype` or `set`). Without it, pandas also creates empty groups for unused categories.

## Things to watch
- **Blank probes**: the `all_ds` rows include the `Blank-*` control probes. Drop them with `df = df[~df['gene'].str.startswith('Blank')]`.
- **Unused categories**: after filtering, a categorical column still lists every possible value. Run `df['supertype'] = df['supertype'].cat.remove_unused_categories()` before plotting or counting.

### Soma Volume quantification

**NOTE:** on the alpha parameter for delaunay 3D volume calculation.

`alpha` controls how tightly the mesh wraps around the points. It ranges from "only very close points are joined" (a small alpha) to the **convex hull**, the smallest convex shape enclosing all points (`alpha=0`, the default, or a very large alpha).

## How it works
1. `delaunay_3d` first fills the space between all the points with tetrahedra (a Delaunay tetrahedralization). With no alpha filtering, these tetrahedra fill the convex hull exactly.
2. Every tetrahedron has a **circumsphere**: the one sphere passing through its 4 corners.
3. With `alpha > 0`, VTK **keeps only the tetrahedra whose circumsphere radius is at most `alpha`**. Larger tetrahedra, which bridge big gaps between points, are removed.
4. Triangles, edges and single points left over from removed tetrahedra are kept if they also pass the test. That's why you saw triangles, lines and vertices in the output.

So **`alpha` is a length, in the same units as your coordinates (µm)**. It answers the question: "how big a gap between points should still count as inside the cell?"

- **`alpha` too small** (below the typical spacing between points): almost nothing is kept, and the mesh falls apart into fragments with volume near 0.
- **`alpha` too large**, or `0`: gaps and concave regions get filled in, and the result is the convex hull.

pyvista 0.49's `delaunay_3d` only takes `alpha`, `tol` and `offset`. Choosing which kinds of leftover pieces to keep (VTK's `AlphaTets`, `AlphaTris`, and so on) requires calling VTK directly. That doesn't matter for volume, since only the tetrahedra have volume.

## What it does on your data
I used 200 real cells from H21.33.038/1218552100, all with at least 12 points spanning at least 2 z-planes. The median distance from a point to its nearest neighbour is **1.1 µm**, and the z-planes are 1.5 µm apart.

| alpha (µm) | median volume ÷ convex-hull volume | cells with zero volume |
|---|---|---|
| 1 | 0.00 | 34% |
| 1.5 | 0.02 | 12% |
| 2 | 0.08 | 4% |
| 3 | 0.23 | 1% |
| 4 | 0.37 | 0% |
| 5 | 0.50 | 0% |
| 6 | 0.61 | 0% |
| 8 | 0.74 | 0% |
| 10 | 0.80 | 0% |
| 0 (no filter) | 0.98 | 0% |

**There is no plateau.** With densely sampled points, you'd expect the volume to level off at the true shape over a range of alpha values. Here it keeps rising steadily up to the convex hull. Each cell has only about 45 points, and they lie in 1.5 µm-spaced planes, which is too sparse to define a concave surface. So the volume you get mostly reflects **your choice of alpha**, not the soma's shape.

## What this means for your analysis
- **Your current adaptive alpha** (`3 ×` each cell's median distance to its 10th neighbour) gave about 0.84 of the hull volume. But because alpha changes with each cell's point density, cells with fewer transcripts get a different alpha. Differences between supertypes could then partly reflect transcript counts rather than soma size.
- **For comparing supertypes, I'd use the convex-hull volume.** It has no parameter, it ranked cells almost identically to your alpha shape (Spearman 0.97), and it's easy to describe in methods text. Its known bias is that it overestimates the volume of concave shapes.
- **If you prefer the alpha shape,** use **one fixed alpha for all cells** (for example, 5 µm) and report it. Show that the ranking of supertypes holds across a few values, such as 4, 6 and 8 µm.
- Either way, keep `n_points` and `n_z` (the number of z-planes) as columns, and check that differences between supertypes remain after accounting for transcript count.

In [ ]:
import anndata 
import glob,os 
import numpy as np 
import pandas as pd 

In [10]:
import anndata
H5AD_PATH = r'/mnt/xuisebnas2/Lab/Bereket_SEA_AD_analysis/SEA_AD_dataset/combined_anndata_object/SEAAD_MTG_MERFISH.2024-12-11.h5ad'
adata = anndata.read_h5ad(H5AD_PATH)

In [ ]:
supertypes = adata.obs['Supertype'].unique().tolist()
def get_suface(pt_cloud):
    from scipy.spatial import cKDTree
    import pyvista as pv 
    cloud = pv.PolyData(pt_cloud)
    # Estimate characteristic spacing
    k = 10
    tree = cKDTree(pt_cloud)
    # distances[:, i] is distance to ith neighbor (0 is self = 0)
    distances, _ = tree.query(pt_cloud, k=k+1)
    char = np.median(distances[:, -1])  # distance to the k-th neighbor

    alpha = 3.0 * char  # tune between ~2–4×


    tet = cloud.delaunay_3d(alpha=alpha)
    surf = tet.triangulate().clean()
    return surf

def get_volume(celltype_):
    cell_volume = []
    ROOT_READ_PATH = r'/mnt/xuisebnas2/Lab/Bereket_SEA_AD_analysis/analysis_result/density_based_clustering_fulldataset_supertype'
    parqute_fls = glob.glob(ROOT_READ_PATH + os.sep + '*' + os.sep + '*.parquet')
    for fl in parqute_fls:
        df = pd.read_parquet(
            fl,
            columns=['global_x','global_y','global_z','gene','cell_id','cell','set','supertype'],
            filters=[('supertype','==',celltype_),('set','==','somatic_ds')]
            )
        # group by cell-ID 
        for cell_,cell_df in df.groupby('cell'):
            surf = get_suface(cell_df[['global_z','global_x','global_y']].values)
            # save as a dataframe and track the cell-ID 
            donorID = fl.split(os.sep)[-2]
            speciemen = fl.split(os.sep)[-1].split('.')[0]
            sc_volm = pd.DataFrame([]).assign(soma_volume = surf.volume,cell =cell_ ,supertype =celltype_,
                                                                                        donor = donorID, speciemen = speciemen)
            cell_volume += [sc_volm]
    cell_volume = pd.concat(cell_volume,ignore_index=True)
    cell_volume[['supertype','donor','speciemen']] = cell_volume[['supertype','donor','speciemen']].astype('category')
    # save as a parquet file 
    out = os.path.dirname(ROOT_READ_PATH) + os.sep + 'somaVolume_perSupertype'
    os.makedirs(out,exist_ok=True)
    cell_volume.to_parquet(out + os.sep + f'celltype_somavolume.parquet', index = False)


['Oligo_1',
 'Astro_3',
 'Astro_1',
 'L2/3 IT_1',
 'Pvalb_15',
 'L5/6 NP_3',
 'Oligo_2',
 'Micro-PVM_2_3-SEAAD',
 'L6b_5',
 'OPC_2_1-SEAAD',
 'Sst_9',
 'L2/3 IT_13',
 'L6 IT_2',
 'Monocyte',
 'L6 CT_3',
 'Pericyte_1',
 'L2/3 IT_5',
 'Vip_4',
 'Pvalb_2',
 'L6 IT Car3_3',
 'L6 IT_1',
 'Vip_16',
 'Lamp5_2',
 'Lamp5_Lhx6_1',
 'L4 IT_2',
 'Sst_10',
 'Vip_19',
 'L4 IT_3',
 'L2/3 IT_10',
 'L6b_4',
 'L2/3 IT_6',
 'L6 CT_4',
 'Chandelier_1',
 'Endo_2',
 'Micro-PVM_3-SEAAD',
 'OPC_1',
 'L4 IT_4',
 'Oligo_2_1-SEAAD',
 'Vip_21',
 'L2/3 IT_2',
 'Vip_9',
 'Pvalb_9',
 'Astro_5',
 'L6b_6',
 'Vip_15',
 'L2/3 IT_12',
 'Vip_11',
 'Oligo_3',
 'Pvalb_5',
 'L5 IT_3',
 'Astro_4',
 'L5 IT_5',
 'Vip_14',
 'Sst Chodl_2',
 'Vip_12',
 'L5 IT_1',
 'Sst_13',
 'Pvalb_13',
 'Lamp5_1',
 'Pax6_1',
 'L2/3 IT_3',
 'Oligo_4',
 'Micro-PVM_2_1-SEAAD',
 'L5 IT_6',
 'VLMC_1',
 'L6 IT Car3_2',
 'L5 IT_7',
 'Endo_1',
 'Lamp5_3',
 'OPC_2',
 'L6 CT_1',
 'L5 IT_2',
 'L4 IT_1',
 'SMC-SEAAD',
 'Astro_2',
 'L6b_2',
 'Sst_3',
 'Sncg_3